# Fine-tune ClinicalBERT for MediScribe NER

This Kaggle notebook fine-tunes `emilyalsentzer/Bio_ClinicalBERT` for token classification and exports a Hugging Face model that MediScribe can load through `MEDISCRIBE_NER_MODEL`.

**Important:** if you do not attach your own dataset, the notebook generates template-based synthetic examples for all MediScribe labels. This is useful for an end-to-end demo, but synthetic-only metrics are not evidence of clinical accuracy; validate on independently annotated, de-identified clinical data before deployment. This notebook does not provide medical advice and model output requires clinician review.

## Dataset format

Option A: add a Kaggle Dataset containing `train.jsonl` and optionally `validation.jsonl`. Each line must be a JSON object with whitespace/tokenized words and BIO labels:

```json
{"tokens": ["Patient", "has", "chest", "pain"], "ner_tags": ["O", "O", "B-symptom", "I-symptom"]}
```

Option B: attach no dataset. The notebook generates synthetic examples for every MediScribe entity type. You can set `USE_PUBLIC_DATASET = True` to add the public `ncbi_disease` dataset, which contributes additional `diagnosis` examples. Synthetic data should not be the final evaluation set.

In [15]:
!pip -q install "transformers>=4.45,<5" "datasets>=3,<4" accelerate

In [16]:
from pathlib import Path
import json, os, random, shutil
from datasets import Dataset, DatasetDict, load_dataset

SEED = 42
random.seed(SEED)

# Change this to the Kaggle input directory containing train.jsonl.
DATA_DIR = Path("/kaggle/input/mediscribe-ner")
TRAIN_FILE = DATA_DIR / "train.jsonl"
VALID_FILE = DATA_DIR / "validation.jsonl"
OUTPUT_DIR = Path("/kaggle/working/clinicalbert-mediscribe")

ENTITY_TYPES = [
    "symptom", "drug", "dosage", "frequency", "duration", "route", "test",
    "adverse_event", "diagnosis", "medical_history", "allergy", "patient", "physician",
]
LABELS = ["O"] + [prefix + "-" + entity for entity in ENTITY_TYPES for prefix in ("B", "I")]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

def normalize_tag(tag):
    tag = str(tag).strip()
    if tag.upper() == "O":
        return "O"
    prefix, separator, entity = tag.partition("-")
    if not separator or prefix.upper() not in {"B", "I"}:
        raise ValueError(f"Invalid BIO tag: {tag!r}")
    entity = entity.lower()
    if entity not in ENTITY_TYPES:
        raise ValueError(f"Unsupported MediScribe entity type: {entity!r}")
    return f"{prefix.upper()}-{entity}"

def read_jsonl(path):
    rows = []
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            row = json.loads(line)
            if len(row["tokens"]) != len(row["ner_tags"]):
                raise ValueError(f"{path}:{line_number}: tokens and ner_tags differ in length")
            rows.append({"tokens": row["tokens"], "ner_tags": [normalize_tag(t) for t in row["ner_tags"]]})
    if not rows:
        raise ValueError(f"No records found in {path}")
    return rows

def make_synthetic_rows(repeats=250):
    templates = [
        (["Patient", "reports", "severe", "chest", "pain"], ["O", "O", "O", "B-symptom", "I-symptom"]),
        (["Prescribe", "amoxicillin", "500", "mg", "twice", "daily"], ["O", "B-drug", "B-dosage", "I-dosage", "B-frequency", "I-frequency"]),
        (["Continue", "the", "medication", "for", "seven", "days"], ["O", "O", "O", "O", "B-duration", "I-duration"]),
        (["Give", "the", "tablet", "by", "mouth"], ["O", "O", "O", "O", "B-route"]),
        (["Order", "a", "complete", "blood", "count"], ["O", "O", "B-test", "I-test", "I-test"]),
        (["The", "patient", "has", "pneumonia"], ["O", "O", "O", "B-diagnosis"]),
        (["Past", "medical", "history", "includes", "diabetes"], ["O", "B-medical_history", "I-medical_history", "O", "I-medical_history"]),
        (["Patient", "is", "allergic", "to", "penicillin"], ["B-patient", "O", "O", "O", "B-allergy"]),
        (["Patient", "experienced", "nausea", "after", "treatment"], ["B-patient", "O", "B-adverse_event", "O", "O"]),
        (["The", "physician", "recommended", "rest"], ["O", "B-physician", "O", "O"]),
    ]
    rows = []
    for _ in range(repeats):
        rows.extend({"tokens": tokens[:], "ner_tags": tags[:]} for tokens, tags in templates)
    random.shuffle(rows)
    split = max(1, int(len(rows) * 0.1))
    return rows[split:], rows[:split]

def load_public_ncbi_disease():
    """Load public disease NER data when the user has no labeled dataset."""
    public = load_dataset("ncbi_disease")
    def convert(split):
        rows = []
        feature = public[split].features["ner_tags"].feature
        for row in public[split]:
            tags = []
            for tag_id in row["ner_tags"]:
                tag = feature.int2str(tag_id)
                if tag.upper() == "O":
                    tags.append("O")
                elif tag.upper().startswith("B-"):
                    tags.append("B-diagnosis")
                else:
                    tags.append("I-diagnosis")
            rows.append({"tokens": row["tokens"], "ner_tags": tags})
        return rows
    return convert("train"), convert("validation")

sample = [
    {"tokens": ["Patient", "has", "chest", "pain"], "ner_tags": ["O", "O", "B-symptom", "I-symptom"]},
    {"tokens": ["Start", "aspirin", "75", "mg", "daily"], "ner_tags": ["O", "B-drug", "B-dosage", "I-dosage", "B-frequency"]},
    {"tokens": ["Order", "a", "complete", "blood", "count"], "ner_tags": ["O", "O", "B-test", "I-test", "I-test"]},
    {"tokens": ["History", "of", "diabetes"], "ner_tags": ["O", "O", "B-medical_history"]},
]
USE_PUBLIC_DATASET = False
if TRAIN_FILE.exists():
    train_rows = read_jsonl(TRAIN_FILE)
    valid_rows = read_jsonl(VALID_FILE) if VALID_FILE.exists() else train_rows[:max(1, len(train_rows) // 10)]
    print("Using the attached Kaggle JSONL dataset.")
else:
    train_rows, valid_rows = make_synthetic_rows()
    print("No local dataset found; using generated synthetic examples for all MediScribe labels.")
    if USE_PUBLIC_DATASET:
        try:
            public_train, public_valid = load_public_ncbi_disease()
            train_rows.extend(public_train)
            valid_rows.extend(public_valid)
            print("Added public NCBI Disease examples for diagnosis.")
        except Exception as exc:
            print(f"Public dataset was not added: {exc}")
dataset = DatasetDict({"train": Dataset.from_list(train_rows), "validation": Dataset.from_list(valid_rows)})
print(dataset)
print(f"Labels: {len(LABELS)} | Training records: {len(train_rows)}")

No local dataset found; using generated synthetic examples for all MediScribe labels.
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2250
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 250
    })
})
Labels: 27 | Training records: 2250


In [17]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
)

def tokenize_and_align_labels(batch):
    encoded = tokenizer(batch["tokens"], truncation=True, max_length=256, is_split_into_words=True)
    all_labels = []
    for index, word_labels in enumerate(batch["ner_tags"]):
        word_ids = encoded.word_ids(batch_index=index)
        previous_word_id = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(label2id[normalize_tag(word_labels[word_id])])
            else:
                # Label continuation subwords as I-entity where possible.
                tag = normalize_tag(word_labels[word_id])
                if tag != "O" and tag.startswith("B-"):
                    tag = "I-" + tag[2:]
                label_ids.append(label2id[tag])
            previous_word_id = word_id
        all_labels.append(label_ids)
    encoded["labels"] = all_labels
    return encoded

tokenized = dataset.map(tokenize_and_align_labels, batched=True, remove_columns=dataset["train"].column_names)
print(tokenized)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/2250 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2250
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 250
    })
})


In [18]:
import numpy as np
import torch
import inspect
from transformers import DataCollatorForTokenClassification, Trainer, TrainingArguments

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

def bio_spans(tags):
    spans = set()
    start, entity = None, None
    for index, tag in enumerate(tags + ["O"]):
        if tag == "O" or tag.startswith("B-") or (tag.startswith("I-") and tag[2:] != entity):
            if start is not None:
                spans.add((start, index, entity))
            start, entity = None, None
        if tag.startswith("B-") or (tag.startswith("I-") and start is None):
            start, entity = index, tag[2:]
    return spans

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    true_predictions, true_labels = [], []
    for prediction_row, label_row in zip(predictions, labels):
        pred_tags, gold_tags = [], []
        for prediction, gold in zip(prediction_row, label_row):
            if gold != -100:
                pred_tags.append(id2label[int(prediction)])
                gold_tags.append(id2label[int(gold)])
        true_predictions.append(pred_tags)
        true_labels.append(gold_tags)
    gold = {(row_id, start, end, entity) for row_id, row in enumerate(true_labels) for start, end, entity in bio_spans(row)}
    predicted = {(row_id, start, end, entity) for row_id, row in enumerate(true_predictions) for start, end, entity in bio_spans(row)}
    true_positive = len(gold & predicted)
    precision = true_positive / len(predicted) if predicted else 0.0
    recall = true_positive / len(gold) if gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}

evaluation_key = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters else "evaluation_strategy"
training_kwargs = {
    "output_dir": "/kaggle/working/clinicalbert-checkpoints",
    "save_strategy": "epoch",
    "learning_rate": 3e-5,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 32,
    "num_train_epochs": 5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1",
    "greater_is_better": True,
    "logging_steps": 25,
    "report_to": "none",
    "seed": SEED,
    "fp16": torch.cuda.is_available(),
}
training_kwargs[evaluation_key] = "epoch"
args = TrainingArguments(
    **training_kwargs,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()
metrics = trainer.evaluate()
print(metrics)

/tmp/ipykernel_180/47320324.py:63: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.588200,0.006528,1.000000,1.000000,1.000000
2,0.008200,0.002274,1.000000,1.000000,1.000000
3,0.004900,0.001573,1.000000,1.000000,1.000000
4,0.003900,0.001313,1.000000,1.000000,1.000000
5,0.003600,0.001237,1.000000,1.000000,1.000000


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

{'eval_loss': 0.006527802906930447, 'eval_precision': 1.0, 'eval_recall': 1.0, 'eval_f1': 1.0, 'eval_runtime': 0.2834, 'eval_samples_per_second': 882.017, 'eval_steps_per_second': 14.112, 'epoch': 5.0}


In [19]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
with (OUTPUT_DIR / "mediscribe_labels.json").open("w", encoding="utf-8") as handle:
    json.dump({"labels": LABELS, "label2id": label2id, "id2label": id2label}, handle, indent=2)

archive = shutil.make_archive("/kaggle/working/clinicalbert-mediscribe", "zip", OUTPUT_DIR)
print(f"Saved model: {OUTPUT_DIR}")
print(f"Download this artifact: {archive}")

Saved model: /kaggle/working/clinicalbert-mediscribe
Download this artifact: /kaggle/working/clinicalbert-mediscribe.zip


In [20]:
from transformers import pipeline

ner = pipeline("token-classification", model=str(OUTPUT_DIR), tokenizer=str(OUTPUT_DIR), aggregation_strategy="simple")
ner("Patient reports severe chest pain and was prescribed aspirin 75 mg daily.")

Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[{'entity_group': 'symptom',
  'score': np.float32(0.7195025),
  'word': 'chest pain',
  'start': 23,
  'end': 33},
 {'entity_group': 'dosage',
  'score': np.float32(0.7437782),
  'word': '75 mg',
  'start': 61,
  'end': 66}]

## Use the exported model in MediScribe

Download and unzip `clinicalbert-mediscribe.zip` into the project's `models/clinicalbert-mediscribe/` directory, install the ML extra, and set:

```powershell
pip install -e ".[ml]"
$env:MEDISCRIBE_NER_MODEL = "models/clinicalbert-mediscribe"
.\start.ps1
```

The model's labels use the same lowercase values as `EntityType` in MediScribe. The downstream medication normalization, safety checks, SOAP generation, and FHIR generation remain unchanged.